# PDelta2-ER2 Long-Context Layer Lab

This notebook tests whether the strongest TinyCeNN single-layer replacement can close the remaining Transformer gap by changing **how the residual error is represented and trained**.

The experiment keeps **Conv4 + PDelta2 F96**, uses only the proven **Residual16** state, trains at **512 tokens**, and tests:

- a raw Residual16 control;
- **rank-8 compressed residual error** on the hardest 25% of head/token positions;
- **rank-12 compressed residual error** on the hardest 25%;
- rank-8 focused on only the hardest 12.5%;
- a learned query gate that predicts where residual correction is needed;
- held-out generalization at **512 / 1024 / 2048 / 4096**.

A strict Transformer quality win requires the paired 95% bootstrap CI of candidate-minus-Transformer NLL to be entirely below zero.


In [ ]:
import os, sys, subprocess, tempfile
from pathlib import Path

assert subprocess.run(["nvidia-smi"], check=False).returncode == 0, "Enable a GPU runtime in Colab."

REPO = Path(tempfile.mkdtemp(prefix="TinyCeNN-er2-"))
subprocess.run(
    ["git", "clone", "--depth", "1", "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO)],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers==4.57.6", "datasets", "huggingface_hub", "pandas", "matplotlib"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "--no-deps"],
    check=True,
)
print("Repository:", REPO)
subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], check=True)


In [ ]:
from datetime import datetime, timezone

PROFILE = "balanced"          # quick | balanced | strong
LAYER = 18
TRAIN_CONTEXT = 512
TEST_CONTEXTS = "512,1024,2048,4096"
SEED = 2026

RESULT_ROOT = Path("/content/TinyCeNN-er2-results")
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT_DIR = RESULT_ROOT / f"{PROFILE}-{stamp}"
print("Output:", OUTPUT_DIR)


In [ ]:
cmd = [
    sys.executable,
    str(REPO / "scripts" / "benchmark_pdelta2_er2_layer.py"),
    "--profile", PROFILE,
    "--layer", str(LAYER),
    "--train-context", str(TRAIN_CONTEXT),
    "--test-contexts", TEST_CONTEXTS,
    "--seed", str(SEED),
    "--output-dir", str(OUTPUT_DIR),
]
print(" ".join(cmd), flush=True)

process = subprocess.Popen(
    cmd,
    cwd=REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end="")
return_code = process.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, cmd)


In [ ]:
import json
import pandas as pd
from IPython.display import display

validation = pd.read_csv(OUTPUT_DIR / "validation_summary.csv")
test = pd.read_csv(OUTPUT_DIR / "test_summary.csv")
effects = pd.read_csv(OUTPUT_DIR / "ingredient_effects.csv")
compressed = pd.read_csv(OUTPUT_DIR / "compressed_residual_summary.csv")
selection = json.loads((OUTPUT_DIR / "selection.json").read_text())
report = json.loads((OUTPUT_DIR / "pdelta2_er2_report.json").read_text())

print("SELECTION")
print(json.dumps(selection, indent=2))

print("\nVALIDATION")
display(validation.sort_values("validation_nll"))

print("\nHELD-OUT TEST")
display(test.sort_values(["context", "delta_nll"]))

print("\nINGREDIENT EFFECTS")
display(effects.sort_values("validation_nll"))

print("\nCOMPRESSED RESIDUAL DIAGNOSTICS")
display(compressed.sort_values("name"))

strict = test[
    (test["candidate"] != "transformer_exact_trainable_control")
    & (test["verdict"] == "strict_quality_win")
]
print("\nSTRICT TRANSFORMER QUALITY WIN:", "YES" if len(strict) else "NO")
if len(strict):
    display(strict)


In [ ]:
import matplotlib.pyplot as plt

plot_df = test[test["candidate"] != "transformer_exact_trainable_control"].copy()
long_names = plot_df.groupby("candidate")["context"].nunique()
long_names = long_names[long_names > 1].index.tolist()

plt.figure(figsize=(9, 5))
for name in long_names:
    part = plot_df[plot_df["candidate"] == name].sort_values("context")
    plt.plot(part["context"], part["delta_nll"], marker="o", label=name)
plt.axhline(0.0, linewidth=1)
plt.axhline(0.02, linewidth=1, linestyle="--")
plt.xlabel("Context length")
plt.ylabel("Candidate - Transformer NLL")
plt.title("PDelta2-ER2 long-context quality gap")
plt.legend(fontsize=8)
plt.grid(alpha=0.25)
plt.show()

plt.figure(figsize=(9, 5))
for name in long_names:
    part = plot_df[plot_df["candidate"] == name].sort_values("context")
    plt.plot(part["context"], 100 * part["state_vs_transformer_fp16"], marker="o", label=name)
plt.xlabel("Context length")
plt.ylabel("Persistent state / Transformer FP16 KV (%)")
plt.title("Persistent-state scaling")
plt.legend(fontsize=8)
plt.grid(alpha=0.25)
plt.show()


In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Downloading complete experiment:", archive)
files.download(archive)
